# Hệ thống Hỏi đáp trên Ảnh (Vietnamese VQA) - Kaggle Runner

Notebook này được thiết kế để chạy tự động các thực nghiệm A1, A2, B1, B2 theo đúng kế hoạch đồ án.

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q transformers peft accelerate bitsandbytes underthesea albumentations rouge_score
import torch
print(f"CUDA available: {torch.cuda.is_available()}")

### 2. Kết nối mã nguồn từ GitHub
Thay thế URL dưới đây bằng địa chỉ kho lưu trữ GitHub của bạn.

In [ ]:
# Thay đổi URL GitHub của bạn tại đây
REPO_URL = "https://github.com/your-username/VQA_VI.git"
!git clone {REPO_URL}
%cd VQA_VI
!ls # Kiểm tra các file đã được tải xuống

In [ ]:
# 3. Kiểm tra dữ liệu (Hãy đảm bảo bạn đã Add Dataset 'openvivqa-vietnamese-vqa')
import os
DATA_PATH = "/kaggle/input/openvivqa-vietnamese-vqa"
if os.path.exists(DATA_PATH):
    print("Dữ liệu đã sẵn sàng!")
    !ls {DATA_PATH}
else:
    print("LỖI: Không tìm thấy dữ liệu. Hãy kiểm tra lại tên Dataset.")

### 4. Huấn luyện Hướng A (Modular Architecture)
Thực hiện huấn luyện A1 (LSTM) và A2 (Transformer).

In [ ]:
# Chạy cấu hình A1
!python train_modular.py --decoder_type lstm --num_epochs 5

In [ ]:
# Chạy cấu hình A2
!python train_modular.py --decoder_type transformer --num_epochs 5

### 5. Huấn luyện Hướng B (Multimodal Pretrained)
Thực hiện Fine-tuning PaliGemma bằng LoRA.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    
    !python train_paligemma.py
except Exception as e:
    print("Vui lòng thiết lập HF_TOKEN trong mục Add-ons -> Secrets.")

### 6. Đánh giá và Xuất báo cáo so sánh

In [ ]:
!python evaluate_all.py